# DINOv2 Refiner v2 — Soft Labels + Partial Unfreeze

### 相比 v1 的核心变更

| 维度 | v1 (昨天) | v2 (本次) |
|------|----------|----------|
| **标签** | 硬标签 (0/1) + HIGH 行级过滤 | 软标签 (prob_*/weight_*/mask_*) 逐类校准 |
| **损失函数** | FocalBCELoss (γ=2, α=0.25) | WeightedSoftBCELoss (逐类权重) |
| **Backbone** | DINOv2 完全冻结 | 最后 N 层解冻 (默认 6) |
| **置信度过滤** | HIGH-only, 丢弃整行 | 无行级过滤, 用 weight 逐类降权 |

### 为什么这些变更能解决过拟合？

1. **软标签**: LLM 说 ACL=1 + HIGH 置信度 → prob=0.80 (不是 1.0)。模型学到"这个可能是撕裂，但不用 100% 相信"
2. **逐类权重**: Effusion LLM 不可靠 → weight≈0.48, 模型知道"积液判断参考就好, 别全信"
3. **解冻 backbone**: DINOv2 最后 6 层从 ImageNet 语义适配到 MRI 病理特征，1.1M → 4.6M 可训练参数, 噪声抵抗能力大幅提升



## 1. 环境安装


In [ ]:
!pip install -q timm pydicom opencv-python scikit-learn



## 2. 导入与配置


In [ ]:
from __future__ import annotations

import gc, math, os, sys, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score, f1_score

print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'GPU count: {torch.cuda.device_count()}')



In [ ]:
# ============================================================
# Configuration -- v2: Soft Labels + Partial Unfreeze
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]

# Soft-label column names (from calibrated CSV)
PROB_COLS   = [f'prob_{c}' for c in TARGET_COLUMNS]
WEIGHT_COLS = [f'weight_{c}' for c in TARGET_COLUMNS]
MASK_COLS   = [f'mask_{c}' for c in TARGET_COLUMNS]

CFG = {
    # --- Paths (Kaggle) ---
    'comp_input':   '/kaggle/input/competitions/rsna-knee-abnormality-detection',
    # v2: upload pseudo_labels_calibrated.csv as a Kaggle Dataset named "rsna-knee-soft-labels"
    'pseudo_input': '/kaggle/input/datasets/easoncyy/rsna-knee-soft-labels',
    'dicom_subdir': 'train_series',
    'output_dir':   '/kaggle/working',

    # --- Data ---
    # ViT attention is O(n²) in patch count. DINOv2 pretrained at 224×224 (256 patches).
    # 280/14=20 → 400 patches (2.4× vs 224). 392→280 cuts attention cost ~4× with
    # negligible detail loss for abnormality detection.
    'image_size': 280,
    'slice_count': 5,
    'center_stride': 3,

    # --- Model ---
    'dinov2_variant': 'vit_small_patch14_dinov2.lvd142m',
    'cls_dim': 384,
    'spa_channels': 64,
    'num_classes': 12,

    # --- v2: Unfreeze strategy ---
    'unfreeze_layers': 4,       # 4 layers: good fine-tuning capacity, less VRAM than 6

    # --- Training ---
    'batch_size': 16,           # safe with image_size=280 + 4 unfrozen layers
    'grad_accum_steps': 1,      # set >1 for larger effective batch without more VRAM
    'epochs': 50,
    'lr': 2e-4,
    'backbone_lr': 1e-5,        # v2: lower LR for unfrozen backbone layers
    'weight_decay': 1e-4,
    'lr_t0': 15,
    'lr_t_mult': 2,
    'lr_eta_min': 1e-6,
    'dropout': 0.2,
    'head_dropout': 0.3,
    'grad_clip': 1.0,
    'early_stop_patience': 10,
    'mixed_precision': True,
    'num_workers': 2,           # fewer workers = less CPU RAM pressure
    'channels_last': True,      # NHWC format speeds up CNN ops (SPA)
    'use_torch_compile': True,   # torch.compile on DINOv2 backbone (~20-30% speedup)
}

# Device setup
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda' if N_GPUS > 0 else 'cpu')
IS_MAIN = True

if IS_MAIN:
    print(f'GPUs: {N_GPUS} | Device: {DEVICE}')
    print(f'--- v2: Soft Labels + Unfreeze {CFG["unfreeze_layers"]} layers ---')
    for k, v in CFG.items():
        print(f'  {k}: {v}')



## 3. 模型组件


In [ ]:
# ============================================================
# 3a. CNN Spatial Pattern Adapter (SPA) -- unchanged from v1
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU()
    def forward(self, x): return self.act(self.bn(self.conv(x)))


class SPAModule(nn.Module):
    """Multi-scale spatial feature extractor."""
    def __init__(self, in_channels=5, base_ch=64):
        super().__init__()
        self.stem = ConvBlock(in_channels, base_ch)
        self.stage1 = nn.Sequential(ConvBlock(base_ch, base_ch), ConvBlock(base_ch, base_ch, 2))
        self.stage2 = nn.Sequential(ConvBlock(base_ch, base_ch*2), ConvBlock(base_ch*2, base_ch*2, 2))
        self.stage3 = nn.Sequential(ConvBlock(base_ch*2, base_ch*4), ConvBlock(base_ch*4, base_ch*4, 2))

    def forward(self, x):
        x = self.stem(x)
        s2 = self.stage1(x)
        s4 = self.stage2(s2)
        s8 = self.stage3(s4)
        return {'s2': s2, 's4': s4, 's8': s8}



In [ ]:
# ============================================================
# 3b. Cross-Modal Fusion -- unchanged from v1
# ============================================================

class CrossModalFusion(nn.Module):
    def __init__(self, cls_dim=384, cnn_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = cls_dim // num_heads
        self.cnn_proj = nn.Linear(cnn_dim, cls_dim)
        self.q_proj = nn.Linear(cls_dim, cls_dim)
        self.k_proj = nn.Linear(cls_dim, cls_dim)
        self.v_proj = nn.Linear(cls_dim, cls_dim)
        self.out_proj = nn.Linear(cls_dim, cls_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(cls_dim)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, cls_token, cnn_features):
        B, C, H, W = cnn_features.shape
        D = cls_token.shape[-1]
        cnn_seq = cnn_features.flatten(2).transpose(1, 2)
        cnn_seq = self.cnn_proj(cnn_seq)
        q = self.q_proj(cls_token).view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(cnn_seq).view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(cnn_seq).view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = self.dropout(attn.softmax(dim=-1))
        out = (attn @ v).transpose(1, 2).contiguous().view(B, D)
        out = self.out_proj(out)
        gate = self.gate.tanh()
        return self.norm(cls_token + gate * out)



In [ ]:
# ============================================================
# 3c. Slice Transformer -- unchanged from v1
# ============================================================

class SliceTransformer(nn.Module):
    def __init__(self, dim=384, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, 5, dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=num_heads, dim_feedforward=dim*4,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.norm = nn.LayerNorm(dim)

    def forward(self, slice_features):
        tokens = slice_features + self.pos_embed
        tokens = self.transformer(tokens)
        return self.norm(tokens.mean(dim=1))



In [ ]:
# ============================================================
# 3d. Classification Head -- unchanged from v1
# ============================================================

class ClassificationHead(nn.Module):
    def __init__(self, in_features=384, hidden=512, num_classes=12, dropout=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x): return self.head(x)



In [ ]:
# ============================================================
# 3e. DINOv2Refiner -- v2: supports partial backbone unfreeze
# ============================================================

class DINOv2Refiner(nn.Module):
    """DINOv2 (partially frozen) + SPA + CrossModalFusion + SliceTransformer + Head.

    v2 changes from v1:
      - unfreeze_layers > 0: unfreezes the last N DINOv2 transformer blocks
      - backbone_lr in optimizer group for lower LR on unfrozen DINOv2 layers
      - trainable parameter count jumps from ~1M to ~4.6M (with unfreeze_layers=6)
    """

    def __init__(self, dinov2_model, spa_channels=64, cls_dim=384,
                 num_slices=5, num_classes=12, num_heads=4,
                 st_layers=2, dropout=0.1,
                 unfreeze_layers=6):
        super().__init__()
        self.num_slices = num_slices
        self.cls_dim = cls_dim
        self.unfreeze_layers = unfreeze_layers

        self.dinov2 = dinov2_model

        # -- v2: Partial unfreeze ----------------------------------
        # DINOv2 ViT-S has 12 blocks: dinov2.blocks[0]..blocks[11]
        # Unfreeze the LAST N blocks + final norm layer
        n_blocks = len(self.dinov2.blocks)

        if unfreeze_layers == 0:
            # v1 behavior: fully frozen
            for p in self.dinov2.parameters():
                p.requires_grad = False
            self.dinov2.eval()
            if IS_MAIN:
                print('[DINOv2] Fully FROZEN (v1 compatibility mode)')
        else:
            # Freeze ALL first, then selectively unfreeze last N blocks
            for p in self.dinov2.parameters():
                p.requires_grad = False

            unfreeze_start = max(0, n_blocks - unfreeze_layers)
            for block in self.dinov2.blocks[unfreeze_start:]:
                for p in block.parameters():
                    p.requires_grad = True

            # Also unfreeze final norm
            if hasattr(self.dinov2, 'norm'):
                for p in self.dinov2.norm.parameters():
                    p.requires_grad = True

            # Keep in eval mode for deterministic DropPath/BatchNorm behavior
            # Parameters still receive gradients -- this is intentional
            self.dinov2.eval()

            trainable_dino = sum(p.numel() for p in self.dinov2.parameters() if p.requires_grad)
            total_dino = sum(p.numel() for p in self.dinov2.parameters())
            if IS_MAIN:
                print(f'[DINOv2] Blocks {unfreeze_start}-{n_blocks-1} UNFROZEN '
                      f'({trainable_dino/1e6:.1f}M / {total_dino/1e6:.1f}M params, '
                      f'{trainable_dino/total_dino*100:.0f}%)')

        # -- Trainable components (unchanged) ----------------------
        self.spa = SPAModule(in_channels=num_slices, base_ch=spa_channels)
        self.fusion = CrossModalFusion(cls_dim=cls_dim, cnn_dim=spa_channels*4,
                                       num_heads=num_heads, dropout=dropout)
        self.slice_transformer = SliceTransformer(dim=cls_dim, num_heads=num_heads,
                                                   num_layers=st_layers, dropout=dropout)
        self.head = ClassificationHead(in_features=cls_dim, hidden=512,
                                       num_classes=num_classes, dropout=CFG['head_dropout'])

        # -- Parameter summary -------------------------------------
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        if IS_MAIN:
            print(f'[DINOv2Refiner] Total: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M '
                  f'({trainable/total*100:.0f}%) | Frozen: {(total-trainable)/1e6:.1f}M')

    def _extract_cls_batched(self, x_3ch):
        """Extract [CLS] tokens from batched 3-channel inputs.

        When unfreeze_layers > 0, runs with gradients (no torch.no_grad).
        When unfreeze_layers == 0, uses torch.no_grad for efficiency.
        """
        if self.unfreeze_layers > 0:
            features = self.dinov2.forward_features(x_3ch)
        else:
            with torch.no_grad():
                features = self.dinov2.forward_features(x_3ch)
        return features[:, 0, :]  # [B_total, cls_dim]

    def forward(self, x):
        B = x.shape[0]

        # -- SPA on 5-channel input ---------------------------------
        spa_features = self.spa(x)

        # -- Batched DINOv2: [B, 5, H, W] -> [B*5, 3, H, W] --------
        x_5bhw = x.permute(1, 0, 2, 3).contiguous()
        x_flat = x_5bhw.view(B * self.num_slices, 1, x.shape[-2], x.shape[-1])
        x_flat_3ch = x_flat.expand(-1, 3, -1, -1)
        all_cls = self._extract_cls_batched(x_flat_3ch)

        all_cls = all_cls.view(self.num_slices, B, self.cls_dim)
        all_cls = all_cls.transpose(0, 1).contiguous()

        # -- Cross-modal fusion per slice ---------------------------
        fused = []
        for i in range(self.num_slices):
            enhanced = self.fusion(all_cls[:, i, :], spa_features['s8'])
            fused.append(enhanced)

        slice_tokens = torch.stack(fused, dim=1)
        study_feature = self.slice_transformer(slice_tokens)
        return self.head(study_feature)

    def train(self, mode=True):
        super().train(mode)
        # DINOv2 stays in eval mode (deterministic DropPath/BatchNorm)
        # but parameters still receive gradients when unfrozen
        self.dinov2.eval()
        return self



In [ ]:
# ============================================================
# 3f. v2: Weighted Soft BCE Loss
# ============================================================
#
# Replaces FocalBCELoss. Key differences:
#   1. Targets are soft probabilities (prob in [0.01, 0.99]), not hard 0/1
#   2. Each class has a training weight based on LLM calibration reliability
#   3. Each class has a mask -- allows ignoring uncertain samples per-class
#
# Loss per element:
#   L = -weight * mask * [prob * log(sigmoid(logit)) + (1-prob) * log(1 - sigmoid(logit))]
#
# Why this fixes overfitting:
#   - Focal: forces model to fit hard labels, penalizing "uncertain" predictions
#   - Soft:  lets model be uncertain where LLM is unreliable (Effusion, Synovitis)
#   - Weight: down-weights unreliable classes rather than discarding the whole row
#   - Mask:   ignores samples where even the calibration is unsure (weight < 0.15)

class WeightedSoftBCELoss(nn.Module):
    """BCE loss with soft probability targets and per-class reliability weights.

    Shape:
        logits:       [B, C]  model output logits
        prob_targets: [B, C]  calibrated probabilities (0.01 ~ 0.99)
        weights:      [B, C]  per-class training weights (0.05 ~ 1.0)
        masks:        [B, C]  binary mask (0=ignore this class for this sample)
    """

    def __init__(self, eps: float = 1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, logits, prob_targets, weights, masks):
        # Clamp targets away from 0/1 for BCE numerical stability
        targets = prob_targets.clamp(self.eps, 1.0 - self.eps)

        # Standard BCE per element
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        # Apply per-class weight and mask
        weighted = bce * weights * masks

        # Normalize by total active mask elements
        denom = masks.sum().clamp(min=1)
        return weighted.sum() / denom


# For comparison / ablation: keep FocalBCELoss available
class FocalBCELoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1.0 - p_t) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_weight * focal_weight * bce).mean()



## 4. DICOM 读取


In [ ]:
# ============================================================
# DICOM I/O helpers (fast path: specific_tags to minimise I/O)
# ============================================================

PLANE_SORT_AXIS = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}

_DICOM_SPECIFIC_TAGS = [
    (0x0020, 0x0032), (0x0020, 0x1041), (0x0020, 0x0013),
    (0x0028, 0x0002), (0x0028, 0x0004), (0x0028, 0x0010),
    (0x0028, 0x0011), (0x0028, 0x0100), (0x0028, 0x0101),
    (0x0028, 0x0102), (0x0028, 0x0103), (0x7FE0, 0x0010),
]

def _get_slice_position(ds, plane=None):
    try:
        ipp = getattr(ds, 'ImagePositionPatient', None)
        if ipp and len(ipp) >= 3:
            axis = PLANE_SORT_AXIS.get(plane, 2) if plane else 2
            return float(ipp[axis])
    except: pass
    try:
        sl = getattr(ds, 'SliceLocation', None)
        if sl is not None: return float(sl)
    except: pass
    try: return float(getattr(ds, 'InstanceNumber', 0))
    except: return 0.0


def read_dicom_series(series_dir, plane=None, image_size=392, lower_pct=0.5, upper_pct=99.5):
    series_dir = Path(series_dir)
    dcm_paths = sorted(series_dir.glob('*.dcm'))
    if not dcm_paths: dcm_paths = sorted(series_dir.glob('*'))

    slices_info = []
    for p in dcm_paths:
        try:
            ds = pydicom.dcmread(str(p), force=True, specific_tags=_DICOM_SPECIFIC_TAGS)
            pos = _get_slice_position(ds, plane)
            img = ds.pixel_array.astype(np.float32)
            slices_info.append((pos, img))
        except: continue

    if not slices_info: raise RuntimeError(f'No DICOM readable: {series_dir}')
    slices_info.sort(key=lambda x: x[0])
    images = np.stack([img for _, img in slices_info], axis=0)

    v_low = np.percentile(images, lower_pct)
    v_high = np.percentile(images, upper_pct)
    images = np.clip(images, v_low, v_high)
    images = (images - v_low) / max(v_high - v_low, 1e-6)

    resized = []
    for img in images:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32)



## 5. 数据集 — 软标签支持


In [ ]:
# ============================================================
# 2.5D Dataset -- v2: returns soft prob/weight/mask alongside images
# ============================================================

class Knee25DSoftLabelDataset(Dataset):
    """2.5D knee MRI dataset with soft pseudo-labels.

    v2 changes from v1:
      - 'labels' DataFrame stores prob_*/weight_*/mask_* instead of hard 0/1
      - __getitem__ returns prob_targets, weights, masks for training
      - No confidence_filter -- uses per-class weights instead of row-level filtering
      - Validation mode returns gold hard labels (for AUC/F1 scoring)
    """

    def __init__(self, series_df, labels_df, dicom_root, image_size=392,
                 slice_count=5, is_train=True, center_stride=3,
                 volume_cache=None, use_soft_labels=True):
        self.dicom_root = Path(dicom_root)
        self.image_size = image_size
        self.slice_count = slice_count
        self.is_train = is_train
        self.half_window = slice_count // 2
        self.volume_cache = volume_cache
        self.use_soft_labels = use_soft_labels and is_train

        # Filter Sagittal T2 FS series
        df = series_df.copy()
        df = df[df['Anatomical_Plane'] == 'Sagittal']
        if 'Fluid_Sensitive' in df.columns: df = df[df['Fluid_Sensitive'] == 1]
        if 'Fat_Suppression' in df.columns: df = df[df['Fat_Suppression'] == 1]

        self.label_df = labels_df
        self.samples = []
        skipped = 0
        for (study_uid, series_uid), grp in df.groupby(['StudyInstanceUID', 'SeriesInstanceUID']):
            if study_uid not in self.label_df.index:
                skipped += 1; continue
            plane = grp.iloc[0]['Anatomical_Plane']
            dicom_dir = self.dicom_root / study_uid / series_uid
            dicom_dir_str = str(dicom_dir)

            if self.volume_cache is not None and dicom_dir_str not in self.volume_cache:
                skipped += 1; continue
            if self.volume_cache is None:
                if not dicom_dir.exists():
                    skipped += 1; continue
                dcm_files = list(dicom_dir.glob('*.dcm'))
                if not dcm_files: dcm_files = list(dicom_dir.glob('*'))
                n_slices = len(dcm_files)
            else:
                n_slices = self.volume_cache[dicom_dir_str].shape[0]

            if n_slices < 3:
                skipped += 1; continue

            for center_idx in range(0, n_slices, center_stride):
                self.samples.append({
                    'study_uid': study_uid, 'series_uid': series_uid,
                    'dicom_dir': dicom_dir_str, 'plane': plane,
                    'center_idx': center_idx, 'n_slices': n_slices,
                })

        if IS_MAIN and skipped:
            print(f'[{type(self).__name__}] {skipped} series skipped')

    def __len__(self): return len(self.samples)

    def _read_volume(self, sample):
        if self.volume_cache is not None:
            volume = self.volume_cache[sample['dicom_dir']]
            is_uint8 = volume.dtype == np.uint8
        else:
            is_uint8 = False
            try:
                volume = read_dicom_series(sample['dicom_dir'], plane=sample['plane'],
                                           image_size=self.image_size)
            except:
                return None, False

        center = sample['center_idx']
        n_total = volume.shape[0]
        half = self.half_window
        indices = [max(0, min(n_total-1, center+o)) for o in range(-half, half+1)]
        stack = volume[indices]

        if is_uint8:
            stack = torch.from_numpy(stack.copy()).float().div_(255.0)
        else:
            stack = torch.from_numpy(stack.copy())
        return stack, True

    def __getitem__(self, idx):
        sample = self.samples[idx]
        study_uid = sample['study_uid']

        image, ok = self._read_volume(sample)
        if not ok:
            image = torch.zeros(self.slice_count, self.image_size, self.image_size)

        label_row = self.label_df.loc[study_uid]

        if self.use_soft_labels:
            # v2: Return soft prob targets + weights + masks
            probs = torch.tensor([float(label_row.get(c, 0.5)) for c in PROB_COLS], dtype=torch.float32)
            weights = torch.tensor([float(label_row.get(c, 0.1)) for c in WEIGHT_COLS], dtype=torch.float32)
            masks = torch.tensor([float(label_row.get(c, 0.0)) for c in MASK_COLS], dtype=torch.float32)
            return {
                'image': image,
                'prob_targets': probs,
                'weights': weights,
                'masks': masks,
                'study_uid': study_uid,
                'plane': sample['plane'],
            }
        else:
            # Validation: return hard labels for scoring
            labels = torch.tensor([float(label_row.get(c, 0.0)) for c in TARGET_COLUMNS], dtype=torch.float32)
            return {
                'image': image,
                'labels': labels,
                'study_uid': study_uid,
                'plane': sample['plane'],
            }



## 6. 加载校准数据


In [ ]:
comp_input = Path(CFG['comp_input'])
pseudo_input = Path(CFG['pseudo_input'])

# Load competition metadata
train_meta = pd.read_csv(comp_input / 'train.csv')
series_meta = pd.read_csv(comp_input / 'train_series.csv')

# Split gold vs unlabeled
label_cols_present = [c for c in TARGET_COLUMNS if c in train_meta.columns]
has_gold = train_meta[label_cols_present].notna().all(axis=1)
gold_df = train_meta[has_gold].copy()

if IS_MAIN:
    print(f'Gold studies: {len(gold_df)}')
    print(f'Total studies: {len(train_meta)}')

# -- v2: Load CALIBRATED pseudo-labels ---------------------------
# pseudo_labels_calibrated.csv columns:
#   StudyInstanceUID | pred_* (12) | conf_* (12) | prob_* (12) | weight_* (12) | mask_* (12)
calibrated_df = pd.read_csv(pseudo_input / 'pseudo_labels_calibrated.csv')

if IS_MAIN:
    print(f'Calibrated pseudo-labels: {len(calibrated_df):,} studies')
    print(f'  Columns ({len(calibrated_df.columns)}): {list(calibrated_df.columns)[:5]}...')

    # -- Quick stats on calibration -----------------------------
    print(f'\n  Calibration summary per class:')
    print(f'  {"Class":<20s} {"Mean Prob":>9s} {"Mean Wgt":>9s} {"%Masked":>8s}')
    print(f'  {"-"*20} {"-"*9} {"-"*9} {"-"*8}')
    for c in TARGET_COLUMNS:
        p_mean = calibrated_df[f'prob_{c}'].mean()
        w_mean = calibrated_df[f'weight_{c}'].mean()
        m_pct = calibrated_df[f'mask_{c}'].mean() * 100
        print(f'  {c:<20s} {p_mean:9.4f} {w_mean:9.4f} {m_pct:7.1f}%')

# Build soft-label DataFrame (for training)
train_labels = calibrated_df[['StudyInstanceUID']].copy()
for c in PROB_COLS:
    train_labels[c] = calibrated_df[c]
for c in WEIGHT_COLS:
    train_labels[c] = calibrated_df[c]
for c in MASK_COLS:
    train_labels[c] = calibrated_df[c]
train_labels = train_labels.set_index('StudyInstanceUID')

# Ensure float32
for c in PROB_COLS + WEIGHT_COLS + MASK_COLS:
    train_labels[c] = pd.to_numeric(train_labels[c], errors='coerce').fillna(0.5 if 'prob' in c else 0.1).astype(np.float32)

# Build hard-label DataFrame (for validation)
val_labels = gold_df[['StudyInstanceUID'] + label_cols_present].copy()
val_labels = val_labels.set_index('StudyInstanceUID')
for c in TARGET_COLUMNS:
    if c not in val_labels.columns:
        val_labels[c] = 0.0
val_labels = val_labels.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

if IS_MAIN:
    print(f'\nTrain studies (soft labels): {len(train_labels):,}')
    print(f'Val studies (gold labels):   {len(val_labels):,}')
    print(f'\n  v2: No row-level confidence filtering -- using per-class weights instead')
    print(f'  v1 used HIGH-only filtering which discarded ~50% of studies')
    print(f'  v2 keeps ALL studies and uses weight_*/mask_* to handle uncertainty')



## 7. 构建 RAM 缓存


In [ ]:
dicom_root = Path(CFG['comp_input']) / CFG['dicom_subdir']
print(f'DICOM root: {dicom_root}')
print(f'DICOM root exists: {dicom_root.exists()}')

def _collect_series_dirs(series_df, labels_df):
    df = series_df.copy()
    df = df[df['Anatomical_Plane'] == 'Sagittal']
    if 'Fluid_Sensitive' in df.columns: df = df[df['Fluid_Sensitive'] == 1]
    if 'Fat_Suppression' in df.columns: df = df[df['Fat_Suppression'] == 1]
    dirs = set()
    for (study_uid, series_uid), grp in df.groupby(['StudyInstanceUID', 'SeriesInstanceUID']):
        if study_uid not in labels_df.index:
            continue
        d = dicom_root / study_uid / series_uid
        if d.exists():
            dirs.add(str(d))
    return dirs

train_dirs = _collect_series_dirs(series_meta, train_labels)
val_dirs = _collect_series_dirs(series_meta, val_labels)
all_dirs = train_dirs | val_dirs

print(f'\nUnique series to cache: {len(all_dirs)}  (train: {len(train_dirs)}, val: {len(val_dirs)})')

VOLUME_CACHE = {}
t_cache = time.time()
failed_series = []
first_success = False

for i, d in enumerate(sorted(all_dirs)):
    try:
        vol = read_dicom_series(d, plane='Sagittal', image_size=CFG['image_size'])
        VOLUME_CACHE[d] = (vol * 255).clip(0, 255).astype(np.uint8)

        if not first_success:
            first_success = True
            elapsed = time.time() - t_cache
            print(f'  OK First series loaded ({vol.shape[0]} slices, {vol.shape[1]}x{vol.shape[2]}) '
                  f'in {elapsed:.0f}s')
    except Exception as e:
        failed_series.append((d, str(e)))
        continue

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t_cache
        ram_gb = sum(v.nbytes for v in VOLUME_CACHE.values()) / 1024**3
        eta_min = (elapsed / (i + 1 - len(failed_series))) * (len(all_dirs) - (i + 1)) / 60
        print(f'  [{i+1:4d}/{len(all_dirs)}] {ram_gb:.1f} GB | {elapsed:.0f}s | ~{eta_min:.0f}min remaining')

cache_time = time.time() - t_cache
ram_gb = sum(v.nbytes for v in VOLUME_CACHE.values()) / 1024**3
print(f'\nVolume cache: {len(VOLUME_CACHE)} series, {ram_gb:.1f} GB in {cache_time:.0f}s')
if failed_series:
    print(f'  {len(failed_series)}/{len(all_dirs)} series failed ({(len(failed_series)/len(all_dirs)*100):.1f}%)')
gc.collect()



In [ ]:
print(f'\n--- Creating datasets (v2: soft labels) ---')

train_ds = Knee25DSoftLabelDataset(series_meta, train_labels, dicom_root,
                                    image_size=CFG['image_size'], slice_count=CFG['slice_count'],
                                    is_train=True, center_stride=CFG['center_stride'],
                                    volume_cache=VOLUME_CACHE, use_soft_labels=True)
val_ds = Knee25DSoftLabelDataset(series_meta, val_labels, dicom_root,
                                  image_size=CFG['image_size'], slice_count=CFG['slice_count'],
                                  is_train=False, center_stride=1,
                                  volume_cache=VOLUME_CACHE, use_soft_labels=False)

if IS_MAIN:
    print(f'Train samples: {len(train_ds):,}  (stride={CFG["center_stride"]})')
    print(f'Val samples:   {len(val_ds):,}  (stride=1)')
    if len(train_ds) == 0:
        print('  WARNING: TRAIN DATASET IS EMPTY!')
    if len(val_ds) == 0:
        print('  WARNING: VAL DATASET IS EMPTY!')

    # Compare with v1: how many more studies do we use?
    # v1 used HIGH-only filtering; v2 keeps all
    train_studies = set(s['study_uid'] for s in train_ds.samples)
    val_studies = set(s['study_uid'] for s in val_ds.samples)
    print(f'  Unique train studies: {len(train_studies):,}')
    print(f'  Unique val studies:   {len(val_studies):,}')



## 8. DataLoader


In [ ]:
loader_kw = dict(num_workers=CFG['num_workers'], pin_memory=True, prefetch_factor=2,
                 persistent_workers=True if CFG['num_workers'] > 0 else False)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, **loader_kw)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, **loader_kw)

if IS_MAIN:
    print(f'Train batches: {len(train_loader):,}  (batch_size={CFG["batch_size"]})')
    print(f'Val batches:   {len(val_loader):,}')



## 9. 构建模型 — 部分解冻 + 分离学习率


In [ ]:
if IS_MAIN: print('Loading DINOv2 backbone...')

dinov2_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=True, num_classes=0,
    img_size=CFG['image_size'],
)

model = DINOv2Refiner(
    dinov2_model=dinov2_backbone,
    spa_channels=CFG['spa_channels'],
    cls_dim=CFG['cls_dim'],
    num_slices=CFG['slice_count'],
    num_classes=CFG['num_classes'],
    num_heads=4, st_layers=2,
    dropout=CFG['dropout'],
    unfreeze_layers=CFG['unfreeze_layers'],  # v2: partial unfreeze
).to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'[Model] DataParallel across {N_GPUS} GPUs')

# torch.compile: graph-level optimization, ~20-30% speedup (PyTorch >= 2.0)
if CFG.get('use_torch_compile', False):
    try:
        if hasattr(model, 'module'):
            model.module.dinov2 = torch.compile(model.module.dinov2, mode='reduce-overhead')
        else:
            model.dinov2 = torch.compile(model.dinov2, mode='reduce-overhead')
        print('[Model] torch.compile enabled on DINOv2 backbone')
    except Exception as e:
        print(f'[Model] torch.compile skipped: {e}')

# -- v2: Separate LR for backbone vs head ------------------------
# Unfrozen DINOv2 layers need lower LR (pretrained weights, fine-tuning)
# SPA/Fusion/SliceTransformer/Head need higher LR (random init)
backbone_params = []
head_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'dinov2' in name:
        backbone_params.append(p)
    else:
        head_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['backbone_lr']},
    {'params': head_params, 'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

if IS_MAIN:
    n_backbone = sum(p.numel() for p in backbone_params)
    n_head = sum(p.numel() for p in head_params)
    print(f'Optimizer: backbone {n_backbone/1e6:.1f}M params @ lr={CFG["backbone_lr"]}')
    print(f'           head     {n_head/1e6:.1f}M params @ lr={CFG["lr"]}')

criterion = WeightedSoftBCELoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG['lr_t0'], T_mult=CFG['lr_t_mult'], eta_min=CFG['lr_eta_min'])

scaler = torch.amp.GradScaler('cuda') if CFG['mixed_precision'] else None



## 10. 训练与验证函数


In [ ]:
# ============================================================
# Training & Validation Functions -- v2: soft label support
# ============================================================

def train_epoch(model, loader, optimizer, criterion, scaler, epoch):
    """v2: uses soft prob targets + weights + masks from dataset.

    Each batch provides:
      - prob_targets: [B, 12] calibrated probabilities
      - weights:      [B, 12] per-class training weights
      - masks:        [B, 12] binary mask (0=skip this class for this sample)

    The loss re-weights uncertain pseudo-labels so the model focuses on
    reliable signals rather than memorizing LLM mistakes.

    Supports gradient accumulation (CFG['grad_accum_steps']) for larger
    effective batch sizes without extra VRAM.
    """
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    use_amp = scaler is not None
    grad_accum = CFG.get('grad_accum_steps', 1)

    for bi, batch in enumerate(loader):
        images = batch['image'].to(DEVICE, non_blocking=True)
        prob_targets = batch['prob_targets'].to(DEVICE, non_blocking=True)
        weights = batch['weights'].to(DEVICE, non_blocking=True)
        masks = batch['masks'].to(DEVICE, non_blocking=True)

        # channels_last conversion for CNN speedup
        if CFG.get('channels_last', False):
            images = images.to(memory_format=torch.channels_last)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, prob_targets, weights, masks)
            loss = loss / grad_accum  # normalize for gradient accumulation

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # Step only after accumulating enough gradients
        if (bi + 1) % grad_accum == 0:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum  # report un-normalized loss

        if IS_MAIN and bi % 20 == 0:
            # Show active mask ratio for monitoring
            active_pct = masks.sum().item() / masks.numel() * 100
            print(f'  Epoch {epoch:3d} [{bi:4d}/{len(loader):4d}] loss={loss.item()*grad_accum:.4f} '
                  f'| active_mask={active_pct:.0f}%', flush=True)

    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(model, loader, criterion, val_batch_size=None):
    """Study-level validation on gold labels.

    Uses standard BCE loss (hard labels) for comparability with v1.
    Aggregates slice-level predictions to study-level via mean pooling.

    val_batch_size: if set, chunks each loader batch into smaller sub-batches
    to reduce peak GPU memory during validation forward pass.
    """
    model.eval()

    study_probs = defaultdict(list)
    study_targets_dict = {}
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        uids = batch['study_uid']

        # Chunked forward pass to limit peak VRAM during validation
        if val_batch_size and images.size(0) > val_batch_size:
            all_logits = []
            for start in range(0, images.size(0), val_batch_size):
                chunk = images[start:start + val_batch_size]
                all_logits.append(model(chunk))
                # Free intermediates immediately
                if start + val_batch_size < images.size(0):
                    torch.cuda.empty_cache()
            logits = torch.cat(all_logits, dim=0)
        else:
            logits = model(images)

        total_loss += F.binary_cross_entropy_with_logits(logits, labels).item()
        n_batches += 1

        probs = torch.sigmoid(logits).cpu().numpy()
        for i, uid in enumerate(uids):
            study_probs[uid].append(probs[i])
            if uid not in study_targets_dict:
                study_targets_dict[uid] = labels[i].cpu().numpy()

    study_uids = list(study_targets_dict.keys())
    n_studies = len(study_uids)
    study_preds = np.zeros((n_studies, 12), dtype=np.float32)
    study_targets = np.zeros((n_studies, 12), dtype=np.float32)

    for i, uid in enumerate(study_uids):
        study_preds[i] = np.mean(study_probs[uid], axis=0)
        study_targets[i] = study_targets_dict[uid]

    per_class = {}
    aucs = []

    for i, c in enumerate(TARGET_COLUMNS):
        y_true = study_targets[:, i]
        y_prob = study_preds[:, i]
        n_pos = int(y_true.sum())

        metrics = {'auc': float('nan'), 'accuracy': float('nan'),
                   'precision': float('nan'), 'recall': float('nan'),
                   'f1': float('nan'), 'n_pos': n_pos, 'n_total': n_studies}

        if n_pos == 0 or n_pos == n_studies:
            y_pred_binary = (y_prob >= 0.5).astype(int)
            metrics['accuracy'] = float((y_true == y_pred_binary).mean())
            per_class[c] = metrics
            continue

        try:
            a = roc_auc_score(y_true, y_prob)
            metrics['auc'] = float(a)
            aucs.append(a)
        except Exception:
            pass

        y_pred_binary = (y_prob >= 0.5).astype(int)
        tp = int(((y_pred_binary == 1) & (y_true == 1)).sum())
        fp = int(((y_pred_binary == 1) & (y_true == 0)).sum())
        fn = int(((y_pred_binary == 0) & (y_true == 1)).sum())
        tn = int(((y_pred_binary == 0) & (y_true == 0)).sum())

        metrics['accuracy'] = float((tp + tn) / n_studies)
        metrics['precision'] = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
        metrics['recall'] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        metrics['f1'] = float(2 * metrics['precision'] * metrics['recall'] /
                              (metrics['precision'] + metrics['recall'])) if (metrics['precision'] + metrics['recall']) > 0 else 0.0
        per_class[c] = metrics

    return {
        'loss': total_loss / max(n_batches, 1),
        'macro_auc': float(np.mean(aucs)) if aucs else 0.0,
        'per_class': per_class,
        'study_preds': study_preds,
        'study_targets': study_targets,
        'study_uids': study_uids,
    }


# -- Per-class threshold analysis (v2 NEW) -----------------------
def analyze_thresholds(val_metrics):
    """Determine whether per-class threshold tuning is needed.

    For each class, sweeps threshold [0.05, 0.95] and finds the optimal
    threshold maximizing F1 score on validation gold labels.

    Interpretation:
      - best_threshold ~= 0.5 for most classes
        -> soft labels fixed the calibration problem -> tuning NOT needed
      - best_threshold << 0.5 for many classes
        -> model is still under-confident -> tuning IS needed
    """
    print(f'\n  {"="*70}')
    print(f'  Per-Class Threshold Analysis')
    print(f'  {"="*70}')
    print(f'  {"Class":<20s} {"Best Thr":>8s} {"F1@0.5":>7s} {"F1@best":>7s} '
          f'{"Pred Mean":>9s} {"%>0.5":>6s} {"Need?":>6s}')
    print(f'  {"-"*20} {"-"*8} {"-"*7} {"-"*7} {"-"*9} {"-"*6} {"-"*6}')

    need_tuning = []
    threshold_table = {}

    for i, c in enumerate(TARGET_COLUMNS):
        y_true = val_metrics['study_targets'][:, i]
        y_prob = val_metrics['study_preds'][:, i]

        if y_true.sum() == 0:
            continue

        best_thr, best_f1 = 0.5, 0.0
        for thr in np.arange(0.05, 0.95, 0.01):
            y_pred = (y_prob >= thr).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr

        f1_default = f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0)
        pct_above = (y_prob > 0.5).mean() * 100
        pred_mean = y_prob.mean()

        needs = 'YES' if best_thr < 0.35 else ('maybe' if best_thr < 0.45 else 'no')
        if best_thr < 0.40:
            need_tuning.append((c, best_thr, f1_default, best_f1))

        threshold_table[c] = {
            'best_threshold': float(best_thr),
            'f1_at_0.5': float(f1_default),
            'f1_at_best': float(best_f1),
            'pred_mean': float(pred_mean),
            'pct_above_0.5': float(pct_above),
        }

        print(f'  {c:<20s} {best_thr:8.2f} {f1_default:7.3f} {best_f1:7.3f} '
              f'{pred_mean:9.4f} {pct_above:5.1f}% {needs:>6s}')

    n_need = len(need_tuning)
    print(f'\n  {n_need}/12 classes need threshold < 0.40')
    if n_need >= 6:
        print(f'  -> Per-class threshold tuning is NECESSARY')
    elif n_need >= 2:
        print(f'  -> Per-class threshold tuning RECOMMENDED for a few classes')
    else:
        print(f'  -> Soft labels fixed calibration! Use default threshold=0.5')

    return threshold_table


def save_validation_report(val_metrics, output_dir, epoch=None, is_best=False, thresholds=None):
    out = Path(output_dir)
    rows = []
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        row = {'class': c, 'auc': m['auc'], 'accuracy': m['accuracy'],
               'precision': m['precision'], 'recall': m['recall'],
               'f1': m['f1'], 'n_pos': m['n_pos'], 'n_total': m['n_total']}
        if thresholds and c in thresholds:
            row['best_threshold'] = thresholds[c]['best_threshold']
            row['f1_at_best'] = thresholds[c]['f1_at_best']
        rows.append(row)

    report_df = pd.DataFrame(rows)
    report_df['macro_auc'] = val_metrics['macro_auc']
    report_df['val_loss'] = val_metrics['loss']

    tag = f'_epoch{epoch}' if epoch else ''
    if is_best: tag = '_best'
    report_path = out / f'validation_report{tag}.csv'
    report_df.to_csv(report_path, index=False)
    print(f'  Report: {report_path}')

    study_rows = []
    for i, uid in enumerate(val_metrics['study_uids']):
        row = {'StudyInstanceUID': uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[f'true_{c}'] = int(val_metrics['study_targets'][i, j])
            row[f'pred_{c}'] = float(val_metrics['study_preds'][i, j])
        study_rows.append(row)
    preds_df = pd.DataFrame(study_rows)
    preds_path = out / f'validation_predictions{tag}.csv'
    preds_df.to_csv(preds_path, index=False)
    print(f'  Predictions: {preds_path} ({len(study_rows)} studies)')

    return report_df


def print_validation_summary(val_metrics):
    print(f'\n  {"Class":<20s} {"AUC":>7s} {"Acc":>7s} {"Prec":>7s} {"Rec":>7s} {"F1":>7s} {"Pos":>5s}')
    print(f'  {"-"*20} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*5}')
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        auc_str = f'{m["auc"]:.3f}' if not math.isnan(m['auc']) else '  N/A  '
        print(f'  {c:<20s} {auc_str:>7s} {m["accuracy"]:7.3f} {m["precision"]:7.3f} '
              f'{m["recall"]:7.3f} {m["f1"]:7.3f} {m["n_pos"]:5d}')
    print(f'  {"-"*20} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*5}')
    print(f'  {"Macro AUC":<20s} {val_metrics["macro_auc"]:7.3f}')
    print()



## 11. 管线检查


In [ ]:
# ============================================================
# Sanity check -- verify soft label pipeline
# ============================================================
if IS_MAIN:
    batch = next(iter(train_loader))
    print(f'Image shape:      {batch["image"].shape}')        # [B, 5, 392, 392]
    print(f'Prob targets:     {batch["prob_targets"].shape}') # [B, 12]
    print(f'Weights:          {batch["weights"].shape}')      # [B, 12]
    print(f'Masks:            {batch["masks"].shape}')        # [B, 12]
    print(f'Study UIDs:       {batch["study_uid"][:3]}')

    p = batch['prob_targets']
    w = batch['weights']
    m = batch['masks']
    print(f'\nSoft label stats (first batch):')
    print(f'  prob   in [{p.min():.3f}, {p.max():.3f}], mean={p.mean():.3f}')
    print(f'  weight in [{w.min():.3f}, {w.max():.3f}], mean={w.mean():.3f}')
    print(f'  mask   % active: {(m == 1).float().mean()*100:.1f}%')
    print(f'  % masked out (weight < 0.15): {(m == 0).float().mean()*100:.1f}%')

    # Forward pass
    with torch.no_grad():
        out = model(batch['image'].to(DEVICE))
    print(f'\nForward pass:')
    print(f'  Output shape: {out.shape}')
    print(f'  Output range: [{out.min().item():.3f}, {out.max().item():.3f}]')

    # Test loss
    loss = criterion(out,
                     batch['prob_targets'].to(DEVICE),
                     batch['weights'].to(DEVICE),
                     batch['masks'].to(DEVICE))
    print(f'  Soft BCE loss: {loss.item():.4f}')

    # Compare with hard-label Focal loss
    hard_labels = (p > 0.5).float()
    focal_loss_fn = FocalBCELoss()
    focal = focal_loss_fn(out, hard_labels.to(DEVICE))
    print(f'  Focal loss (same batch, hard labels): {focal.item():.4f}')
    print(f'  -> Soft loss should be LOWER (uncertain samples down-weighted)')
    print(f'\n  Pipeline OK -- ready for training!')



## 12. 训练循环


In [ ]:
if IS_MAIN:
    steps_per_epoch = len(train_loader)
    eff_batch = CFG['batch_size'] * N_GPUS * CFG.get('grad_accum_steps', 1)
    print(f'\n{"="*60}')
    print(f'v2 Training -- Soft Labels + Unfreeze {CFG["unfreeze_layers"]} DINOv2 layers')
    print(f'Batch={CFG["batch_size"]} × {N_GPUS} GPUs × {CFG.get("grad_accum_steps",1)} accum = {eff_batch} eff')
    print(f'Image={CFG["image_size"]}² | Patches={(CFG["image_size"]//14)**2} | '
          f'Train samples={len(train_ds):,} | Steps={steps_per_epoch:,}')
    print(f'Train studies: {len(set(s["study_uid"] for s in train_ds.samples)):,}')
    print(f'Val studies:   {len(val_ds):,}')
    print(f'{"="*60}\n')

best_auc = 0.0
best_epoch = 0
patience = 0
ckpt_dir = Path(CFG['output_dir']) / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
t_start = time.time()

history = []

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, epoch)

    # Free GPU memory before validation (training fragments VRAM over 1h+ runs)
    torch.cuda.empty_cache()
    gc.collect()

    val_metrics = validate_epoch(model, val_loader, criterion, val_batch_size=CFG['batch_size'] // 2)

    torch.cuda.empty_cache()
    scheduler.step()

    if IS_MAIN:
        epoch_time = time.time() - t0
        elapsed = time.time() - t_start
        vram = torch.cuda.max_memory_allocated(DEVICE) / 1024**3
        torch.cuda.reset_peak_memory_stats(DEVICE)
        lr_now = optimizer.param_groups[0]['lr']

        print(f'\n-- Epoch {epoch:3d}/{CFG["epochs"]} --')
        print(f'  Train Loss: {train_loss:.4f}  |  Val Loss: {val_metrics["loss"]:.4f}')
        print(f'  Val Macro AUC: {val_metrics["macro_auc"]:.4f}  |  LR: {lr_now:.2e}')
        print(f'  Time: {epoch_time:.0f}s epoch | {elapsed/60:.0f}min total | VRAM: {vram:.1f}GB')

        print_validation_summary(val_metrics)

        # v2: Per-class threshold check every 5 epochs
        if epoch % 5 == 0 or epoch == 1:
            threshold_table = analyze_thresholds(val_metrics)

        history.append({
            'epoch': epoch, 'train_loss': train_loss,
            'val_loss': val_metrics['loss'], 'macro_auc': val_metrics['macro_auc'],
        })

        # -- Checkpoint on improvement --
        current_auc = val_metrics['macro_auc']

        if current_auc > best_auc + 0.0005:
            best_auc = current_auc
            best_epoch = epoch
            patience = 0
            state = model.module.state_dict() if N_GPUS > 1 else model.state_dict()
            ckpt_path = ckpt_dir / 'best_model.pt'
            torch.save({'epoch': epoch, 'model': state, 'auc': best_auc, 'config': CFG}, ckpt_path)
            print(f'  >> Best model saved (AUC={best_auc:.4f})')

            # Full threshold analysis on best model
            threshold_table = analyze_thresholds(val_metrics)
            save_validation_report(val_metrics, CFG['output_dir'], epoch=epoch, is_best=True,
                                   thresholds=threshold_table)
        else:
            patience += 1
            if patience >= CFG['early_stop_patience']:
                print(f'\n  Early stopping triggered at epoch {epoch}')
                break

# -- Final Report --
if IS_MAIN:
    total_time = time.time() - t_start
    print(f'\n{"="*60}')
    print(f'v2 Training Complete')
    print(f'  Soft Labels + Unfreeze {CFG["unfreeze_layers"]} DINOv2 layers')
    print(f'  Best Val Macro AUC: {best_auc:.4f} (epoch {best_epoch})')
    print(f'  Total Time:         {total_time/3600:.1f} hours')
    print(f'{"="*60}')

    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(CFG['output_dir']) / 'training_history.csv', index=False)
    print(f'\nOutput files in {CFG["output_dir"]}/:')
    print(f'  training_history.csv')
    print(f'  checkpoints/best_model.pt')
    print(f'  validation_report_best.csv')
    print(f'  validation_predictions_best.csv')
    print(f'{"="*60}')



## 13. 阈值决策


### 训练完成后，根据数据判断是否需要逐类阈值

下面这个 cell 会自动分析每个类别的最优决策阈值：

- **大部分类的最优阈值 ≈ 0.5**：说明软标签已经修复了模型的校准问题 → **不需要逐类阈值**，直接用默认 0.5 即可
- **仍有很多类的最优阈值 < 0.35**：说明模型仍然偏保守 → **需要逐类阈值**，cell 会自动输出每类的最优值

不用猜，让数据说话。



In [ ]:
# ============================================================
# Final Per-Class Threshold Analysis
# ============================================================
if IS_MAIN:
    preds_path = Path(CFG['output_dir']) / 'validation_predictions_best.csv'
    if preds_path.exists():
        preds = pd.read_csv(preds_path)

        print('=' * 70)
        print('FINAL THRESHOLD ANALYSIS (Best Epoch)')
        print('=' * 70)

        summary = []
        for c in TARGET_COLUMNS:
            y_true = preds[f'true_{c}'].values
            y_prob = preds[f'pred_{c}'].values
            n_pos = int(y_true.sum())
            if n_pos == 0: continue

            best_thr, best_f1 = 0.5, 0.0
            for thr in np.arange(0.01, 0.99, 0.01):
                y_pred = (y_prob >= thr).astype(int)
                f1 = f1_score(y_true, y_pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_thr = f1, thr

            f1_default = f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0)
            pred_mean = y_prob.mean()
            pct_05 = (y_prob > 0.5).mean() * 100

            need = 'YES' if best_thr < 0.35 else ('maybe' if best_thr < 0.45 else 'no')
            summary.append({
                'class': c, 'best_threshold': best_thr,
                'f1_at_0.5': f1_default, 'f1_at_best': best_f1,
                'pred_mean': pred_mean, 'pct_above_0.5': pct_05,
                'need_tuning': need,
            })

        summary_df = pd.DataFrame(summary)
        print(summary_df.to_string(index=False))

        n_need = (summary_df['need_tuning'] == 'YES').sum()
        n_maybe = (summary_df['need_tuning'] == 'maybe').sum()

        print(f'\n{"="*70}')
        print(f'VERDICT')
        print(f'{"="*70}')

        if n_need >= 6:
            print(f'{n_need}/12 classes NEED threshold tuning, {n_maybe} maybe.')
            print(f'RECOMMENDATION: Apply per-class optimal thresholds.')
            print(f'\nOptimal thresholds to use:')
            for _, row in summary_df.iterrows():
                if row['need_tuning'] in ('YES', 'maybe'):
                    print(f'  {row["class"]:<20s}: thr={row["best_threshold"]:.2f} '
                          f'(F1: {row["f1_at_0.5"]:.3f} -> {row["f1_at_best"]:.3f})')
        elif n_need >= 2 or n_maybe >= 3:
            print(f'{n_need} need tuning, {n_maybe} maybe. Mostly fixed by soft labels.')
            print(f'RECOMMENDATION: Optional per-class thresholds for the lagging classes.')
        else:
            print(f'Only {n_need} class(es) need tuning.')
            print(f'RECOMMENDATION: Soft labels FIXED the calibration. Use default threshold=0.5.')
            print(f'No per-class threshold tuning needed!')

        summary_df.to_csv(Path(CFG['output_dir']) / 'threshold_analysis.csv', index=False)
        print(f'\nAnalysis saved: threshold_analysis.csv')
    else:
        print('No validation predictions found. Run training first.')

